# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

***Finding A — Random Forest feature importance for Health Score (page 27)***

**What it claims**: Average Position (43%), Impressions (32%), and Scroll Depth (15%) are the strongest predictors of Health Score — together with CTR (8%), they account for ~98% of the model's importance.

**Where the label comes from**: Health Score isn't an independently measured outcome — it's a hand-built composite (page 5): Impressions (30pts) + Position (30pts) + CTR (20pts) + Scroll Depth (20pts). The Random Forest is then trained to predict this composite using, among other things, those same four raw inputs.

**My methodology question**: Since the target is a deterministic combination of the top four "predictive" features, doesn't the importance ranking mostly describe the scoring formula rather than revealing what independently drives real search outcomes? The paper calls this relationship "partly" circular — I'd ask what the same Random Forest would show if retrained on an outcome that isn't built from these inputs, like next-month clicks or session growth. That would separate "the model can reconstruct its own recipe" from "the model found something new.


***Finding B — Growth-prediction logistic regression, 71% holdout accuracy***

**What it claims**: A logistic regression predicting growing-vs-declining pages reaches 71% accuracy on an 80/20 holdout split.

**Where the label comes from**: The growth/decline label comes from the same 61.8K-page active-content sample used throughout the ML appendix.

**My methodology question**: The paper doesn't say how the split was made — row-level or brand-grouped — or report the base rate, or confirm all 57 brands appear on both sides. A row-level split lets the model see other pages from the same brand during training, so 71% could partly reflect "this smells like Brand X" rather than a general growth pattern. I ran this exact comparison on my own Week 5 model: switching from a random split to a client-grouped split dropped Precision@10 from 0.82 to 0.64 — a large gap from changing nothing but the split method. Without knowing which split the paper used, 71% is hard to interpret as a real signal versus an inflated one.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
#load previous data
!pip install -q duckdb huggingface_hub

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
base = "hf://datasets/FlyRank/internship-warehouse"

def load_month_agg(month_str):
    return con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS gsc_impressions,
               SUM(gsc_clicks) AS gsc_clicks,
               SUM(gsc_sum_position) AS gsc_sum_position,
               SUM(ga4_sessions) AS ga4_sessions,
               SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
        FROM read_parquet('{base}/fact_content_daily_performance/month={month_str}/data_0.parquet')
        GROUP BY client_hash_id, content_hash_id
    """).df()

df_feb_agg = load_month_agg('2026-02')
df_march_agg = load_month_agg('2026-03')
df_april_agg = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS gsc_clicks_apr
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-04/data_0.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Feb: {len(df_feb_agg)}, March: {len(df_march_agg)}, April(clicks only): {len(df_april_agg)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feb: 321546, March: 331437, April(clicks only): 362172


In [6]:
def position_bucket(pos):
    if pd.isna(pos):
        return 'no_position_data'
    elif pos <= 3:
        return '1-3 (top)'
    elif pos <= 10:
        return '4-10'
    elif pos <= 20:
        return '11-20'
    else:
        return '21+'

In [7]:
def build_rule_df(raw_agg):
    d = raw_agg.copy()
    d['gsc_avg_position'] = d['gsc_sum_position'] / d['gsc_impressions']
    d.loc[d['gsc_sum_position'] == 0, 'gsc_avg_position'] = pd.NA

    signal = d[d['gsc_impressions'] > 0].copy()
    signal['ctr'] = signal['gsc_clicks'] / signal['gsc_impressions']
    signal['position_bucket'] = signal['gsc_avg_position'].apply(position_bucket)
    peer_ctr = signal.groupby('position_bucket')['ctr'].mean().to_dict()

    d['ctr'] = d['gsc_clicks'] / d['gsc_impressions']
    d['engagement_rate'] = d['ga4_engaged_sessions'] / d['ga4_sessions']
    d['position_bucket'] = d['gsc_avg_position'].apply(position_bucket)
    d['peer_avg_ctr'] = d['position_bucket'].map(peer_ctr)

    eligible = (d['gsc_impressions'] >= 50) | (d['ga4_sessions'] >= 10)
    return d[eligible].copy()

march_rule_df = build_rule_df(df_march_agg)
feb_rule_df = build_rule_df(df_feb_agg)
print(f"March-eligible: {len(march_rule_df)}, Feb-eligible: {len(feb_rule_df)}")

March-eligible: 116512, Feb-eligible: 93871


In [8]:
MIN_CLICKS_FOR_LABEL = 5
DECLINE_THRESHOLD = 0.8

def build_labelable(rule_df, next_month_clicks_df, next_month_col):
    merged = rule_df.merge(next_month_clicks_df, on=['client_hash_id', 'content_hash_id'], how='left')
    tracked = merged[next_month_col].notna()
    lab = merged[tracked & (merged['gsc_clicks'] >= MIN_CLICKS_FOR_LABEL)].copy()
    lab['decline_label'] = (lab[next_month_col] < DECLINE_THRESHOLD * lab['gsc_clicks']).astype(int)
    return lab

labelable = build_labelable(march_rule_df, df_april_agg, 'gsc_clicks_apr')  # march -> april (existing)

mar_clicks_only = df_march_agg[['client_hash_id', 'content_hash_id', 'gsc_clicks']].rename(
    columns={'gsc_clicks': 'gsc_clicks_mar'})
feb_labelable = build_labelable(feb_rule_df, mar_clicks_only, 'gsc_clicks_mar')  # feb -> march (new)

print(f"March->April labelable: {len(labelable)}, base rate {labelable['decline_label'].mean():.3f}")
print(f"Feb->March labelable:   {len(feb_labelable)}, base rate {feb_labelable['decline_label'].mean():.3f}")

March->April labelable: 28805, base rate 0.545
Feb->March labelable:   21773, base rate 0.313


In [9]:
FEATURES = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions',
            'ga4_engaged_sessions', 'ctr', 'engagement_rate', 'peer_avg_ctr']

def build_X(df, feature_cols):
    X = df[feature_cols].copy()
    worst = X['gsc_avg_position'].max()
    X['gsc_avg_position'] = X['gsc_avg_position'].fillna(worst + 10 if pd.notna(worst) else 100)
    X['ga4_sessions'] = X['ga4_sessions'].fillna(0)
    X['ga4_engaged_sessions'] = X['ga4_engaged_sessions'].fillna(0)
    X['engagement_rate'] = X['engagement_rate'].fillna(0)
    X['ctr'] = X['ctr'].fillna(0)
    X['peer_avg_ctr'] = X['peer_avg_ctr'].fillna(X['peer_avg_ctr'].median())
    X = X.join(pd.get_dummies(df['position_bucket'], prefix='pos', drop_first=True))
    return X

def precision_at_k(ids, scores, labels, k):
    temp = pd.DataFrame({'content_hash_id': ids, 'score': scores, 'label': labels})
    ranked = temp.sort_values(['score', 'content_hash_id'], ascending=[False, True])
    return ranked.head(k)['label'].mean()

X_full = build_X(labelable, FEATURES)
y = labelable['decline_label'].values
groups = labelable['client_hash_id'].values
print(f"X_full: {X_full.shape}, base rate {y.mean():.3f}")

X_full: (28805, 11), base rate 0.545


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.